In [1]:
# imports
try:
    import pandas as pd
    import numpy as np
    from tqdm import tqdm
    from toponymy import Toponymy, ToponymyClusterer
    import temporalmapper as tm
    from sentence_transformers import SentenceTransformer
    import torch
except ImportError as e:
    print(f"Failed to import: {e}")
    print(f"Attempting to install packages...")
    %pip install pandas numpy tqdm toponymy temporal-mapper sentence_transformers torch ipywidgets jupyter tokenizers

from chronoscope import *
from graphing_utilities import *

In [5]:
"""
Parameters
"""
# Set this to where you want to output the temporal topic model
data_path = "UNGDC-demo-small" 
pickled_topic_model = "ToponyMapper-demo-small.pkl"

# Set this to agree with `UNGDC-DataPrep.ipynb`
input_parquet = data_path+"/UNGDC-demo-30pc-dataset.pq"

# If you're not using a Cohere LLM, you will need to modify the LLM wrapper
# in the toponymy set up cell
api_key_file = 'cohere.txt'


# Throw away some data if needed to make things quicker
decimate = False
decimation_factor = 4


## Mapper settings
## See the temporal mapper parameter selection documentation 
## https://temporal-mapper.readthedocs.io/en/latest/paramselection.html
## Note that the clusterer will be overwritten by Toponymy's clusterer.
mapper_params = {
    "N_checkpoints": 14, #!=== Most important parameter -- number of time bins 
    "neighbours": 500, 
    "slice_method": "data", 
    "overlap":0.8
}

## Toponymy settings
compute_names = True
# if false, much quicker but no text names for topics
model_name = 'all-mpnet-base-v2' # embedding model for sentence transformers

toponymy_object_description = "excerpts from a speech"
toponymy_corpus_description = "United Nations General Debate Transcripts"

toponymy_exemplar_method = "central"
toponymy_keyphrase_method = "information_weighted"
toponymy_subtopic_method = "facility_location"

### THE CLUSTERER PARAMS DOMINATE RUNTIME -- MORE CLUSTERS IS EXPONENTIALLY SLOWER ###
# we really want every time step to have the same number of layers so adjust this if needed to make that happen
clusterer_params = {
    'min_clusters':4,
    'max_layers':3,
    'base_min_cluster_size':150,
    'next_cluster_size_quantile':0.8,
    'verbose':False
}


## == not needed to change from here down == ##
configuration_dict = {
    'input_parquet':input_parquet,
    'decimate':decimate,
    'decimation_factor':decimation_factor,
    'mapper_params':mapper_params,
    'model_name':model_name,
    'toponymy_object_description':toponymy_object_description,
    'toponymy_corpus_description':toponymy_corpus_description,
    'toponymy_exemplar_method':toponymy_exemplar_method,
    'toponymy_keyphrase_method':toponymy_keyphrase_method,
    'toponymy_subtopic_method':toponymy_subtopic_method,
    'clusterer_params':clusterer_params,
}

import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
def print_ts(txt):
    now = datetime.now(ZoneInfo('US/Eastern'))
    formatted_datetime = now.strftime("%Y-%m-%d %H:%M:%S")
    print(formatted_datetime+": "+txt)

try:
    Path(data_path).mkdir(parents=True, exist_ok=True)
    with open(data_path+'/configuration.json', 'w') as f:
        json.dump(configuration_dict, f, indent=4)
    print_ts(f"Created experimental folder {data_path}, and saved configuation file.")
except Exception as e:
    print_ts('Error:', e)


import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from tqdm.auto import tqdm
# Register tqdm with pandas
tqdm.pandas()

# Import the pre-processed dataset (see: un-data-exploration.ipynb)
df = pd.read_parquet(input_parquet)

embedding_vectors = df['embedding'].to_numpy()
reduced_vectors = df['reduced'].to_numpy()
text = df['chunk'].to_numpy()
time = df['year'].to_numpy()

rng = np.random.default_rng()
if decimate:
    choice_index = rng.choice(len(df), size=len(df)//decimation_factor, replace=False)
    embedding_vectors = embedding_vectors[choice_index]
    reduced_vectors = reduced_vectors[choice_index]
    text = text[choice_index]
    time = time[choice_index]

# Sort all arrays by time
sorted_idx = np.argsort(time)
time = time[sorted_idx]
text = text[sorted_idx]
reduced_vectors = np.vstack(reduced_vectors[sorted_idx])
embedding_vectors = np.vstack(embedding_vectors[sorted_idx])

2026-02-11 08:36:02: Created experimental folder UNGDC-demo-small, and saved configuation file.


In [6]:
## Toponymy Setup
from toponymy.toponymy import Toponymy, ToponymyClusterer
from toponymy.cluster_layer import ClusterLayerText  
from toponymy.llm_wrappers import AzureAINamer, AsyncAzureAINamer
api_key_file = 'cohere.txt'
with open(api_key_file, 'r') as file:
    api_key = file.read().strip()
# Initialize Cohere wrapper  
llm=AzureAINamer(
    api_key, 
    endpoint="https://azureaitimcuse5821437469.services.ai.azure.com/models",
    model="Cohere-command-r-08-2024",
)
  
# Test connection  
llm.test_llm_connectivity()  

from sentence_transformers import SentenceTransformer
import torch
# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print_ts(f"Using device: {device}")

# Load the embedding model
# Options: 'all-MiniLM-L6-v2' (fast, 384 dim), 'all-mpnet-base-v2' (better quality, 768 dim)
model_name = 'all-mpnet-base-v2'
model = SentenceTransformer(model_name, device=device)
print_ts(f"Loaded model: {model_name}")

clusterer = ToponymyClusterer(**clusterer_params)

toponymy_params = {
    'llm_wrapper':llm,
    'text_embedding_model':model,
    'clusterer':clusterer,
    'object_description':"excerpts from a speech",
    'corpus_description':"United Nations General Debate Transcripts",
    'exemplar_delimiters':["<EXAMPLE_TRANSCRIPT>\n","\n</EXAMPLE_TRANSCRIPT>\n\n"],
}
toponymy_fit_params = {
    'exemplar_method':toponymy_exemplar_method,
    'keyphrase_method':toponymy_keyphrase_method,
    'subtopic_method':toponymy_subtopic_method,
}


2026-02-11 08:36:11: Using device: cpu
2026-02-11 08:36:12: Loaded model: all-mpnet-base-v2


In [7]:
import temporalmapper.weighted_clustering as tmwc
import pickle

dp = Path(data_path+"/pickled")
dp.mkdir(parents=True, exist_ok=True)
mapper_params['kernel'] = tmwc.square

print_ts("#===== Starting temporal topic model =====#")
filename = pickled_topic_model
p = Path(data_path+"/pickled/"+filename)

print_ts(f"Computing temporal Mapper graph")
scope = Chronoscope(
    time,
    reduced_vectors,
    text,
    embedding_vectors,
    mapper_params,
    toponymy_params,
    toponymy_fit_params=toponymy_fit_params,
    verbose=True
)
scope.slice()
scope.cluster()
for i in range(scope.n_layers):
    print_ts(f"Computing graph for layer {i}")
    scope.compute_graph(i)
if compute_names:
    scope.compute_toponymies()
    try:
        scope.connect_topics_to_nodes()
    except Exception as e:
        print(e)

try:
    fp = data_path+"/pickled"
    scope.save(fp)
except exception as e:
    print(f"Save failed: {e}")

2026-02-11 08:36:12: #===== Starting temporal topic model =====#
2026-02-11 08:36:12: Computing temporal Mapper graph
Computing k nearest neighbours...
Computing spatial density...


Toponymy clustering each time slice: 100%|██████████| 14/14 [00:03<00:00,  3.76it/s]


2026-02-11 08:36:35: Computing graph for layer 0


Converting clusters to vertices: 100%|██████████| 14/14 [00:00<00:00, 1743.22it/s]


793 vertices added.


...

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/3 [00:00<?, ?layer/s]

Finding previous topic names: 100%|██████████| 9/9 [00:00<00:00, 1640.25it/s]


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Building topic names by layer:   0%|          | 0/3 [00:00<?, ?layer/s]

In [ ]:
from chronoscope import Chronoscope

scope = Chronoscope.load(
    data_path+"/pickled"
)


In [ ]:
scope.temporal_plot(1, layout_optimization='barycenter')